In [1]:
# Cell 3: imports
import warnings
warnings.filterwarnings('ignore')

import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import OneHotEncoder,TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.feature_selection import SelectKBest, f_regression

import joblib

print("All imports loaded.")


All imports loaded.


In [3]:
data = pd.read_parquet('D:/Study/data_science/Appliance Intelligence Price Tracker and Recommender/data/03.cleaned/multi_appliances_cleaned_engineered.parquet')
print(f"Shape: {data.shape}")
print(f"Columns: {list(data.columns)}")
print()
print("Dtypes summary:")
print(data.dtypes.value_counts())
print()
print("First 3 rows:")
print(data.head(3))
print()
print("Missing values per column (top 5):")
print(data.isnull().sum().sort_values(ascending=False).head(5))

Shape: (2870, 67)
Columns: ['price', 'rating', 'category', 'n_features', 'brand_name', 'star_rating', 'has_star_rating', 'has_inverter', 'has_wifi', 'has_voice_control', 'has_app_control', 'smart_connectivity_score', 'ac_split', 'ac_window', 'ac_pm25_filter', 'ac_hepa_filter', 'ac_auto_clean', 'ac_hot_and_cold', 'ac_copper_condenser', 'ac_Dehumidification', 'ac_Turbo Mode', 'ac_Self Diagnosis', 'ref_single_door', 'ref_multi_door', 'ref_chest_freezer', 'ref_side_door', 'ref_french_door', 'ref_double_door', 'ref_triple_door', 'ref_frost_free', 'ref_convertible', 'ref_door_alarm', 'ref_door_lock', 'ref_dispenser', 'ref_door_display', 'ref_mini', 'wm_fully_automatic', 'wm_semi_automatic', 'wm_with_dryer', 'wm_washer_only', 'wm_dryer_only', 'wm_front_load', 'wm_top_load', 'wm_inbuilt_heater', 'wm_quick_wash', 'wm_ss_tub', 'wm_child_lock', 'wm_shock_proof', 'wm_display', 'log_price', 'capacity_sq', 'n_features_sq', 'rating_sq', 'capacity_n_features', 'capacity_rating', 'rating_n_features', '

In [4]:
# Split the Feature and Target as DataFrame and Series from the data
X = data.drop(columns=['price','log_price'])
y = data['log_price']
print(f"Shape X -> {X.shape}")
print(f"Shape y -> {y.shape}")

Shape X -> (2870, 65)
Shape y -> (2870,)


In [154]:
X.info()

<class 'pandas.DataFrame'>
Index: 2870 entries, 0 to 2924
Data columns (total 65 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   rating                    2870 non-null   float64
 1   category                  2870 non-null   str    
 2   n_features                2870 non-null   int64  
 3   brand_name                2870 non-null   str    
 4   star_rating               2870 non-null   float64
 5   has_star_rating           2870 non-null   int64  
 6   has_inverter              2870 non-null   int64  
 7   has_wifi                  2870 non-null   int64  
 8   has_voice_control         2870 non-null   int64  
 9   has_app_control           2870 non-null   int64  
 10  smart_connectivity_score  2870 non-null   int64  
 11  ac_split                  2870 non-null   Int8   
 12  ac_window                 2870 non-null   Int8   
 13  ac_pm25_filter            2870 non-null   Int8   
 14  ac_hepa_filter          

### ***Baseline Linear Regression***

In [155]:
# OHE Category
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Columns to encode
categorical_cols = ['category','brand_name']

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            categorical_cols
        )
    ],
    remainder='passthrough'
)

print(f"Raw X shape: {X.shape}")
print(f"Train shape: {X_train.shape}")
print(f"Test shape:  {X_test.shape}")
print()

print("Categorical columns to OHE:")
print(categorical_cols)

print("\nAll remaining columns are passed through unchanged.")
print("\nPreprocessor defined. It will be fit only on the training data.")

Raw X shape: (2870, 65)
Train shape: (2296, 65)
Test shape:  (574, 65)

Categorical columns to OHE:
['category', 'brand_name']

All remaining columns are passed through unchanged.

Preprocessor defined. It will be fit only on the training data.


In [156]:
# ====================================================================
# Train baseline linear regression + 5-fold CV
# ====================================================================
# CV gives a more honest score than a single 80/20 split because it
# averages across 5 different train/val partitions. Each row appears
# in val exactly once, in train 4 times.
#
# Metrics:
#   - R²    : fraction of price variance explained (1.0 = perfect)
#   - RMSE  : average prediction error in rupees (lower = better)
#   - MAE   : median-ish error in rupees (lower = better; less
#             sensitive to a few big misses than RMSE)

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold

from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_r2 = cross_val_score(
    pipe,
    X_train,
    y_train,
    cv=kf,
    scoring='r2'
)

cv_rmse = -cross_val_score(
    pipe,
    X_train,
    y_train,
    cv=kf,
    scoring='neg_root_mean_squared_error'
)

cv_mae = -cross_val_score(
    pipe,
    X_train,
    y_train,
    cv=kf,
    scoring='neg_mean_absolute_error'
)
print(f"5-fold CV on training set (n={len(y_train)}):")
print(f"R²    : {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")
print(f"RMSE  : {cv_rmse.mean():.4f} ± {cv_rmse.std():.4f} (log-price)")
print(f"MAE   : {cv_mae.mean():.4f} ± {cv_mae.std():.4f} (log-price)")
print()
print(f"Per-fold R²: {cv_r2}")

5-fold CV on training set (n=2296):
R²    : 0.8810 ± 0.0299
RMSE  : 0.1792 ± 0.0181 (log-price)
MAE   : 0.1246 ± 0.0037 (log-price)

Per-fold R²: [0.88790629 0.89486643 0.82277719 0.89127308 0.90834961]


In [157]:
# Let us check if we drop brand_name column and just use brand_frequency 
# OHE Category
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train = X_train.drop('brand_name',axis=1)
X_test = X_test.drop('brand_name',axis=1)

# Columns to encode
categorical_cols = ['category']

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            categorical_cols
        )
    ],
    remainder='passthrough'
)

print(f"Raw X shape: {X.shape}")
print(f"Train shape: {X_train.shape}")
print(f"Test shape:  {X_test.shape}")
print()

print("Categorical columns to OHE:")
print(categorical_cols)

print("\nAll remaining columns are passed through unchanged.")
print("\nPreprocessor defined. It will be fit only on the training data.")

# Fitting the model Again

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_r2 = cross_val_score(
    pipe,
    X_train,
    y_train,
    cv=kf,
    scoring='r2'
)

cv_rmse = -cross_val_score(
    pipe,
    X_train,
    y_train,
    cv=kf,
    scoring='neg_root_mean_squared_error'
)

cv_mae = -cross_val_score(
    pipe,
    X_train,
    y_train,
    cv=kf,
    scoring='neg_mean_absolute_error'
)
print(f"5-fold CV on training set (n={len(y_train)}):")
print(f"R²    : {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")
print(f"RMSE  : {cv_rmse.mean():.4f} ± {cv_rmse.std():.4f} (log-price)")
print(f"MAE   : {cv_mae.mean():.4f} ± {cv_mae.std():.4f} (log-price)")
print()
print(f"Per-fold R²: {cv_r2}")

Raw X shape: (2870, 65)
Train shape: (2296, 64)
Test shape:  (574, 64)

Categorical columns to OHE:
['category']

All remaining columns are passed through unchanged.

Preprocessor defined. It will be fit only on the training data.
5-fold CV on training set (n=2296):
R²    : 0.8453 ± 0.0398
RMSE  : 0.2043 ± 0.0209 (log-price)
MAE   : 0.1469 ± 0.0048 (log-price)

Per-fold R²: [0.86174988 0.86284587 0.76662909 0.85895582 0.87629767]


In [158]:
# ====================================================================
# Train on full training set, evaluate on test set (the "real" score)
# ====================================================================
# This is the model that "ships". It has never seen the test set.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train = X_train.drop('brand_name',axis=1)
X_test = X_test.drop('brand_name',axis=1)

# Columns to encode
categorical_cols = ['category']

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            categorical_cols
        )
    ],
    remainder='passthrough'
)

from sklearn.metrics import root_mean_squared_error

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

pipe.fit(X_train, y_train)

y_train_pred = np.expm1(pipe.predict(X_train))
y_test_pred  = np.expm1(pipe.predict(X_test))
y_train_true = np.expm1(y_train)
y_test_true = np.expm1(y_test)

train_r2   = r2_score(y_train_true, y_train_pred)
train_rmse = root_mean_squared_error(y_train_true, y_train_pred)
train_mae  = mean_absolute_error(y_train_true, y_train_pred)

test_r2   = r2_score(y_test_true, y_test_pred)
test_rmse = root_mean_squared_error(y_test_true, y_test_pred)
test_mae  = mean_absolute_error(y_test_true, y_test_pred)

print(f"Train (n={len(y_train)}):")
print(f"  R²    : {train_r2:.4f}")
print(f"  RMSE  : ₹{train_rmse:,.0f}")
print(f"  MAE   : ₹{train_mae:,.0f}")
print()
print(f"Test (n={len(y_test)}):")
print(f"  R²    : {test_r2:.4f}")
print(f"  RMSE  : ₹{test_rmse:,.0f}")
print(f"  MAE   : ₹{test_mae:,.0f}")
print()
print(f"Gap (train R² − test R²): {train_r2 - test_r2:+.4f}")


Train (n=2296):
  R²    : 0.7920
  RMSE  : ₹7,819
  MAE   : ₹4,764

Test (n=574):
  R²    : 0.8137
  RMSE  : ₹7,358
  MAE   : ₹4,678

Gap (train R² − test R²): -0.0217


**Overall Assessment**
- The model demonstrates good generalization ability.
- There is no sign of overfitting or underfitting due to train-test mismatch.
- The use of a log-transformed target helped stabilize variance and improve predictive performance.
- A test R² of around 0.63 indicates that the available features capture a substantial portion of price variation, although additional informative features could further improve accuracy.

**Final Conclusion**

- The final Linear Regression model achieved a test R² of 0.629, MAE of ₹6,894, and RMSE of ₹12,581, with a negligible train-test performance gap of −0.0037. These results indicate a stable and well-generalized model capable of providing reasonably accurate price estimates for unseen products. The model explains approximately 63% of the variability in product prices, making it a strong baseline for this price prediction task.

### ***Hyper parameter Tuned Linear Models***

***Using SelectKBest TO get best 30 Numeric Features + Categorical Features***

In [159]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train = X_train.drop('brand_name',axis=1)
X_test = X_test.drop('brand_name',axis=1)

# Universal features
categorical_cols = ['category']

# Binary engineered features
int8_cols = X_train.select_dtypes(include=['Int8']).columns.tolist()

# Continuous variables
continuous_cols = [
    col for col in X_train.columns
    if col not in categorical_cols + int8_cols
]

# Columns to encode
categorical_cols = ['category']


In [160]:
#train_and_evaluate() helper
# Wraps a model in the preprocessor pipeline, runs GridSearchCV,
# evaluates in rupee space, returns a results dict for comparison.

def train_and_evaluate(
    name,
    model,
    param_grid,
    cv=5,
    verbose=True
):
    """
    Train a model with GridSearchCV, evaluate in rupee space, return metrics.

    Returns a dict with: name, best_params, fit_time, cv_score_log,
    train/test metrics, predictions, and the fitted best model.
    """
    # Wrap in pipeline
    preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            categorical_cols
        ),
        (
            'cont',
            'passthrough',
            continuous_cols
        ),
        (
            'int8',
            Pipeline([
                ('select', SelectKBest(f_regression, k=20))
            ]),
            int8_cols
        )
    ]
)

    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # Grid search (CV in log space, default R²)
    grid = GridSearchCV(
        pipe,
        param_grid,
        cv=cv,
        scoring='r2',
        n_jobs=-1,
        refit=True
    )
    grid.fit(X_train, y_train)

    # Best model
    best_model = grid.best_estimator_
    best_params = grid.best_params_

    # Predict in log space, convert to rupee space
    y_train_pred = np.expm1(best_model.predict(X_train))
    y_test_pred = np.expm1(best_model.predict(X_test))
    y_train_true = np.expm1(y_train)
    y_test_true = np.expm1(y_test)

    # Metrics in rupee space
    train_r2 = r2_score(y_train_true, y_train_pred)
    test_r2 = r2_score(y_test_true, y_test_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train_true, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test_true, y_test_pred))
    train_mae = mean_absolute_error(y_train_true, y_train_pred)
    test_mae = mean_absolute_error(y_test_true, y_test_pred)
    gap = train_r2 - test_r2

    if verbose:
        print(f"\n{'='*60}")
        print(f"  {name}")
        print(f"{'='*60}")
        print(f"Best params: {best_params}")
        print(f"CV best R² (log space): {grid.best_score_:.4f}")
        print(f"\n  Train (₹): R²={train_r2:.4f}, RMSE=₹{train_rmse:,.0f}, MAE=₹{train_mae:,.0f}")
        print(f"  Test  (₹): R²={test_r2:.4f}, RMSE=₹{test_rmse:,.0f}, MAE=₹{test_mae:,.0f}")
        print(f"  Train-test gap: {gap:+.4f}")

    return {
        'name': name,
        'best_params': best_params,
        'cv_score_log': grid.best_score_,
        'train_r2': train_r2, 'test_r2': test_r2,
        'train_rmse': train_rmse, 'test_rmse': test_rmse,
        'train_mae': train_mae, 'test_mae': test_mae,
        'gap': gap,
        'y_train_pred': y_train_pred,
        'y_test_pred': y_test_pred,
        'y_train_true': y_train_true,
        'y_test_true': y_test_true,
        'model': best_model,
    }

# Container to collect all model results
results = []

print("Helper function defined. Ready to train models.")


Helper function defined. Ready to train models.


***1. Linear Regression***

In [161]:

model_lr = LinearRegression()

param_grid_lr = {
    'model__fit_intercept': [True, False],
}

result_lr = train_and_evaluate(
    name='LinearRegression',
    model=model_lr,
    param_grid=param_grid_lr,
    cv=5
)

results.append(result_lr)



  LinearRegression
Best params: {'model__fit_intercept': True}
CV best R² (log space): 0.8423

  Train (₹): R²=0.7367, RMSE=₹8,797, MAE=₹4,892
  Test  (₹): R²=0.8108, RMSE=₹7,414, MAE=₹4,785
  Train-test gap: -0.0741


***2. Ridge Regression(L2 Regularization)***

In [162]:
# L2 shrinks coefficients toward zero — useful for handling multicollinearity
# Tune alpha to find the right shrinkage strength.

model_ridge = Ridge()

param_grid_ridge = {
    'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
    'model__fit_intercept': [True, False],
}

result_ridge = train_and_evaluate(
    name='Ridge (L2)',
    model=model_ridge,
    param_grid=param_grid_ridge,
    cv=5
)

results.append(result_ridge)



  Ridge (L2)
Best params: {'model__alpha': 1.0, 'model__fit_intercept': True}
CV best R² (log space): 0.8425

  Train (₹): R²=0.7379, RMSE=₹8,778, MAE=₹4,888
  Test  (₹): R²=0.8101, RMSE=₹7,427, MAE=₹4,789
  Train-test gap: -0.0723


***3. Lasso (L1) Regularization***

In [163]:
# L1 can zero out features entirely — useful for noise reduction and
# automatic feature selection. Note: smaller alpha range than Ridge
# because L1 shrinkage is more aggressive per unit alpha.

model_lasso = Lasso(max_iter=10000)

param_grid_lasso = {
    'model__alpha': [0.0001, 0.001, 0.005, 0.01, 0.05, 0.1],
    'model__fit_intercept': [True, False],
}

result_lasso = train_and_evaluate(
    name='Lasso (L1)',
    model=model_lasso,
    param_grid=param_grid_lasso,
    cv=5
)

results.append(result_lasso)



  Lasso (L1)
Best params: {'model__alpha': 0.001, 'model__fit_intercept': True}
CV best R² (log space): 0.8439

  Train (₹): R²=0.7348, RMSE=₹8,829, MAE=₹4,921
  Test  (₹): R²=0.8076, RMSE=₹7,478, MAE=₹4,801
  Train-test gap: -0.0728


***4. ElasticNet (L1 + L2 mix)***

In [164]:
# l1_ratio controls the mix: 0 = pure Ridge, 1 = pure Lasso.
# Given that neither pure Ridge nor pure Lasso helped, ElasticNet is
# unlikely to break out — but we test it for completeness.

model_en = ElasticNet(max_iter=10000)

param_grid_en = {
    'model__alpha': [0.001, 0.01, 0.1],
    'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9],
}

result_en = train_and_evaluate(
    name='ElasticNet (L1+L2)',
    model=model_en,
    param_grid=param_grid_en,
    cv=5
)

results.append(result_en)



  ElasticNet (L1+L2)
Best params: {'model__alpha': 0.001, 'model__l1_ratio': 0.9}
CV best R² (log space): 0.8439

  Train (₹): R²=0.7356, RMSE=₹8,815, MAE=₹4,914
  Test  (₹): R²=0.8080, RMSE=₹7,469, MAE=₹4,798
  Train-test gap: -0.0724


### **Tree Based Models**
- I would keep Brand_name, and No Feature Selection Needed

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state= 42
)
# Save the splits to the artifacts folder

# Save the splits to the artifacts folder
# X_train and X_test are already DataFrames, so this works
X_train.to_parquet('D:/Study/data_science/Appliance Intelligence Price Tracker and Recommender/models/dataset/X_train.parquet')
X_test.to_parquet('D:/Study/data_science/Appliance Intelligence Price Tracker and Recommender/models/dataset/X_test.parquet')

# Convert y_train and y_test to DataFrames before saving
y_train.to_frame().to_parquet('D:/Study/data_science/Appliance Intelligence Price Tracker and Recommender/models/dataset/y_train.parquet')
y_test.to_frame().to_parquet('D:/Study/data_science/Appliance Intelligence Price Tracker and Recommender/models/dataset/y_test.parquet')

# Universal features
categorical_cols = ['category']


In [173]:
#train_and_evaluate() helper
# Wraps a model in the preprocessor pipeline, runs GridSearchCV,
# evaluates in rupee space, returns a results dict for comparison.

def train_and_evaluate(
    name,
    model,
    param_grid,
    cv=5,
    verbose=True
):
    """
    Train a model with GridSearchCV, evaluate in rupee space, return metrics.

    Returns a dict with: name, best_params, fit_time, cv_score_log,
    train/test metrics, predictions, and the fitted best model.
    """
    # Wrap in pipeline
    preprocessor = ColumnTransformer(
        transformers=[
        (
            'target_enc', 
            TargetEncoder(), 
            ['brand_name'] 
        ),
        (
            'ohe',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            categorical_cols # Now only contains 'category'
        )
    ],
    remainder='passthrough'
)

    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # Grid search (CV in log space, default R²)
    grid = GridSearchCV(
        pipe,
        param_grid,
        cv=cv,
        scoring='r2',
        n_jobs=-1,
        refit=True
    )
    grid.fit(X_train, y_train)

    # Best model
    best_model = grid.best_estimator_
    best_params = grid.best_params_

    # Predict in log space, convert to rupee space
    y_train_pred = np.expm1(best_model.predict(X_train))
    y_test_pred = np.expm1(best_model.predict(X_test))
    y_train_true = np.expm1(y_train)
    y_test_true = np.expm1(y_test)

    # Metrics in rupee space
    train_r2 = r2_score(y_train_true, y_train_pred)
    test_r2 = r2_score(y_test_true, y_test_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train_true, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test_true, y_test_pred))
    train_mae = mean_absolute_error(y_train_true, y_train_pred)
    test_mae = mean_absolute_error(y_test_true, y_test_pred)
    gap = train_r2 - test_r2

    if verbose:
        print(f"\n{'='*60}")
        print(f"  {name}")
        print(f"{'='*60}")
        print(f"Best params: {best_params}")
        print(f"CV best R² (log space): {grid.best_score_:.4f}")
        print(f"\n  Train (₹): R²={train_r2:.4f}, RMSE=₹{train_rmse:,.0f}, MAE=₹{train_mae:,.0f}")
        print(f"  Test  (₹): R²={test_r2:.4f}, RMSE=₹{test_rmse:,.0f}, MAE=₹{test_mae:,.0f}")
        print(f"  Train-test gap: {gap:+.4f}")

    return {
        'name': name,
        'best_params': best_params,
        'cv_score_log': grid.best_score_,
        'train_r2': train_r2, 'test_r2': test_r2,
        'train_rmse': train_rmse, 'test_rmse': test_rmse,
        'train_mae': train_mae, 'test_mae': test_mae,
        'gap': gap,
        'y_train_pred': y_train_pred,
        'y_test_pred': y_test_pred,
        'y_train_true': y_train_true,
        'y_test_true': y_test_true,
        'model': best_model,
    }

# Container to collect all model results
results = []

print("Helper function defined. Ready to train models.")


Helper function defined. Ready to train models.


***5. Decision Tree Regressor***

In [174]:
# First non-linear model. Trees can capture interactions like
# Tune max_depth (tree complexity) and min_samples_leaf (regularization).

model_dt = DecisionTreeRegressor(random_state=42)

param_grid_dt = {
    'model__max_depth': [3, 5, 7, 10],
    'model__min_samples_leaf': [1, 5, 20],
}

result_dt = train_and_evaluate(
    name='DecisionTree',
    model=model_dt,
    param_grid=param_grid_dt,
    cv=5
)

results.append(result_dt)



  DecisionTree
Best params: {'model__max_depth': 10, 'model__min_samples_leaf': 5}
CV best R² (log space): 0.8688

  Train (₹): R²=0.8910, RMSE=₹5,659, MAE=₹3,499
  Test  (₹): R²=0.8584, RMSE=₹6,415, MAE=₹4,106
  Train-test gap: +0.0327


***6. Random Forest Regressor***

In [175]:
# Bagging ensemble of trees. Averaging reduces variance, so we expect
# a lift over DecisionTree — but the one-hot brand issue is structural
# and won't fully resolve here. XGBoost (next cell) has native
# categorical handling and is the real test.

model_rf = RandomForestRegressor(random_state=42, n_jobs=-1)

param_grid_rf = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_leaf': [1, 5],
}

result_rf = train_and_evaluate(
    name='RandomForest',
    model=model_rf,
    param_grid=param_grid_rf,
    cv=3
)

results.append(result_rf)



  RandomForest
Best params: {'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}
CV best R² (log space): 0.8949

  Train (₹): R²=0.9264, RMSE=₹4,652, MAE=₹2,835
  Test  (₹): R²=0.8831, RMSE=₹5,829, MAE=₹3,624
  Train-test gap: +0.0433


***7. Gradient Boosting***

In [176]:
from sklearn.ensemble import GradientBoostingRegressor

# Gradient Boosting
model_gb = GradientBoostingRegressor(random_state=42)

# Small grid (~24 combinations)
param_grid_gb = {
    'model__n_estimators': [100, 300],
    'model__learning_rate': [0.05, 0.1],
    'model__max_depth': [3, 5],
    'model__min_samples_leaf': [1, 5, 20],
}

result_gb = train_and_evaluate(
    name='GradientBoosting',
    model=model_gb,
    param_grid=param_grid_gb,
    cv=5
)

results.append(result_gb)


  GradientBoosting
Best params: {'model__learning_rate': 0.05, 'model__max_depth': 5, 'model__min_samples_leaf': 1, 'model__n_estimators': 300}
CV best R² (log space): 0.9048

  Train (₹): R²=0.9479, RMSE=₹3,914, MAE=₹2,415
  Test  (₹): R²=0.8801, RMSE=₹5,903, MAE=₹3,502
  Train-test gap: +0.0678


***8. XGBoost Regressor***

In [177]:
model_xgb = XGBRegressor(
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    enable_categorical=False,
)

param_grid_xgb = {
    'model__n_estimators': [200, 500],
    'model__max_depth': [4, 6],
    'model__learning_rate': [0.05, 0.1],
    'model__min_child_weight': [1, 5],
}

result_xgb = train_and_evaluate(
    name='XGBoost',
    model=model_xgb,
    param_grid=param_grid_xgb,
    cv=3
)

results.append(result_xgb)



  XGBoost
Best params: {'model__learning_rate': 0.1, 'model__max_depth': 4, 'model__min_child_weight': 1, 'model__n_estimators': 200}
CV best R² (log space): 0.9024

  Train (₹): R²=0.9290, RMSE=₹4,567, MAE=₹2,786
  Test  (₹): R²=0.8834, RMSE=₹5,821, MAE=₹3,482
  Train-test gap: +0.0457


### ***Saving The Best Model***

In [178]:
#  save the best model

# Find the best result by test R²
best_result = max(results, key=lambda r: r['test_r2'])
best_model = best_result['model']

print(f"Best model: {best_result['name']}")
print(f"Test R²:   {best_result['test_r2']:.4f}")
print(f"Test MAE:  ₹{best_result['test_mae']:,.0f}")
print(f"Test RMSE: ₹{best_result['test_rmse']:,.0f}")
print(f"Best params: {best_result['best_params']}")

# Save
model_dir = Path(r"d:\Study\data_science\underpriced-listing-predictor\models\saved_models")
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / "multi_category_xgb_pipeline.pkl"

joblib.dump(best_model, model_path)

# Verify by reloading and predicting on 3 sample rows
loaded = joblib.load(model_path)
sample_features = X_train.head(3)  # raw features (pipeline handles one-hot internally)
sample_pred_log = loaded.predict(sample_features)
sample_pred_rupee = np.exp(sample_pred_log)
sample_actual_rupee = np.exp(y_train.iloc[:3].values)

print(f"\nVerification (loaded model, 3 sample predictions):")
for i in range(3):
    pred = sample_pred_rupee[i] if hasattr(sample_pred_rupee, '__getitem__') else sample_pred_rupee.iloc[i]
    print(f"  Row {i}: predicted ₹{pred:,.0f}  vs  actual ₹{sample_actual_rupee[i]:,.0f}")

print(f"\nSaved to: {model_path}")
print(f"File size: {model_path.stat().st_size / 1024:.1f} KB")


Best model: XGBoost
Test R²:   0.8834
Test MAE:  ₹3,482
Test RMSE: ₹5,821
Best params: {'model__learning_rate': 0.1, 'model__max_depth': 4, 'model__min_child_weight': 1, 'model__n_estimators': 200}

Verification (loaded model, 3 sample predictions):
  Row 0: predicted ₹17,371  vs  actual ₹19,991
  Row 1: predicted ₹30,473  vs  actual ₹28,300
  Row 2: predicted ₹43,353  vs  actual ₹41,990

Saved to: d:\Study\data_science\underpriced-listing-predictor\models\saved_models\multi_category_xgb_pipeline.pkl
File size: 322.9 KB
